# Negative Binomial Model for Cyclist Volume in Adelaide

See `adelaide-model-2/data-notes.md` for discussion on decisions regarding variables and methods used.

## Variables:
- POW_CTW: N people who work in this DZN and cycle to work
- POW_FE: N people who work in this DZN and are fullly engaged in employment or study
- PD_MB: MB population density
- DNL_DZN: Distance to nearest large DZN
- M_30_40, F_15_25: Age demographics, try males 30-40 and females 15-25 (best from Dadshova's)
- H_3500: Households with a THI of > 3500, place of enumeration, MB
- INF_CR: Listed cycle routes

__Monash Bicycle Infrastructure Classification__ - Binary dummy variables for every type of bicycle infrastructure

- MBI_BP: Dedicated bike path, Shared bike path
- MBI_PP: Pedestrian path/street with cycling allowed
- MBI_PL: Protected bike lane
- MBI_BL: Buffered bike lane (kerb-side), Buffered bike lane (road-side), Buffered bike lane (both sides)
- MBI_PTL: Painted bike lane, Painted bike lane (single-side)
- MBI_Ad: Advisory bike lane, Advisory bike lane (single-side), Peak hour advisory bike lane, Peak hour advisory bike lane (single-side)
- MBI_PkPt: Peak hour painted bike lane, Peak hour painted bike lane (single-side)
- MBI_Sh: Sharrow
- MBI_SZ: Shared zone
- MBI_Bus: Bus lane with cycling allowed
- MBI_Mx: Mixed traffic
- MBI_No: Bicyclists dismount, Pedestrian path/street with cycling not allowed

## Correlations

There is bound to be a lot of correlation between many of the variables chosen for this model - eg. the Monash Classifiers. The goal of this model is to identify the best variables to use for the final model.

__*Retrieve generated graph*__

See `./GRAPH_SETUP.ipynb` for graph network generation. Graph comes with all infrastructure features from OpenStreetMap, all listed cycle routes, Mesh Blocks and Destination Zones. Mesh blocks and destination zones are loaded into __*each node*__, so MB and DZN-related variables must be calculated for (u,v) of each edge. Depending on the variable, the minimum, maximum, or average of (u,v) should be taken as the final number for that edge.

In [ ]:
import osmnx as ox

# Load graph from GeoPackage
gpkg_path = "./graph_cache/adelaide_bike_graph.gpkg"
G = ox.io.load_graph_geopackage(gpkg_path)
print(f"Loaded graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

# Verify key attributes exist
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True)
mbi_cols = [c for c in edges.columns if c.startswith('MBI_') or c == 'IS_CYCLE_ROUTE']
print(f"Infrastructure columns found: {mbi_cols}")
print(f"MBI columns: {[c for c in edges.columns if c.startswith('MBI_')]}")
print(f"IS_CYCLE_ROUTE present: {'IS_CYCLE_ROUTE' in edges.columns}")

## Retrieve variables for graph and regression table

__Generic Census Data:__
- DNL_DZN
- PD_MB

__Census Table Data:__
- POW_CTW
- POW_FE
- M_30_40
- F_15_25
- H_3500

__Bicycle Infrastructure Data:__
- MBI_BP
- MBI_PP
- MBI_PL
- MBI_BL
- MBI_PTL
- MBI_Ad
- MBI_PkPt
- MBI_Sh
- MBI_SZ
- MBI_Bus
- MBI_Mx
- MBI_No

__*DNL_DZN*__

Distance to nearest large destination zone (8000+ jobs). These zones include the areas near North Terrace / Rundle / Franklin st, the RAH, between Pirie and Wakefield St, East End. 8000 is an arbitrary number.

__*PD_MB*__

Population density by mesh block

__*POW_CTW*__

N people who work in this DZN and cycle to work

__*POW_FE*__

N people who work in this DZN and are fullly engaged in employment or study

__*PD_MB*__

MB population density

__*DNL_DZN*__

Distance to nearest large DZN

__*M_30_40, F_15_25*__
Age demographics, try males 30-40 and females 15-25 (best from Dadshova's)

__*H_3500*__

Households with a THI of > 3500, place of enumeration, MB

__*INF_CR*__

Listed cycle routes

__*MBI classifiers*__

All Monash classifiers are retrieved via `infrastructure/get_bike_infra.py`.

In [ ]:
from infrastructure.get_bike_infra import enrich_graph_with_monash_classifiers
G = enrich_graph_with_monash_classifiers(G)